In [ ]:

!pip install -q autogluon.tabular
import json,time
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import balanced_accuracy_score
from autogluon.tabular import TabularPredictor
ID="id";TARGET="health_condition";C=["at-risk","fit","unhealthy"]
base=Path("/kaggle/input/competitions/playground-series-s6e7");tr=pd.read_csv(base/"train.csv");te=pd.read_csv(base/"test.csv");t0=time.time()
def fe(d):
 z=d.drop(columns=[ID],errors="ignore").copy();s=z.sleep_duration
 z["sleep_lt6"]=np.where(s.isna(),np.nan,(s<6).astype(float));z["sleep_lt7"]=np.where(s.isna(),np.nan,(s<7).astype(float))
 z["key_missing_count"]=z[["sleep_duration","stress_level","physical_activity_level"]].isna().sum(1)
 z["rule_unhealthy"]=((s<6)&z.stress_level.eq("high")).astype(float);z["rule_fit"]=((s>=7)&z.stress_level.eq("low")&z.physical_activity_level.eq("active")).astype(float)
 return z
A=fe(tr);B=fe(te)
predictor=TabularPredictor(label=TARGET,problem_type="multiclass",eval_metric="balanced_accuracy",path="/kaggle/working/ag_model",verbosity=2)
# OOF and test inference both require the fitted fold children to remain on
# disk. Discarding them saves space but makes predict_proba() fail afterward.
predictor.fit(A,presets="medium_quality",time_limit=10800,num_bag_folds=5,num_stack_levels=1,dynamic_stacking=False,save_bag_folds=True)
o=predictor.predict_proba_oof();p=predictor.predict_proba(B)
o=o.reindex(columns=C);p=p.reindex(columns=C);score=float(balanced_accuracy_score(tr[TARGET],o.idxmax(1)))
od=pd.DataFrame({ID:tr[ID]});td=pd.DataFrame({ID:te[ID]})
for c in C:od[c]=o[c].to_numpy();td[c]=p[c].to_numpy()
od.to_csv("oof_preds.csv",index=False);td.to_csv("test_preds.csv",index=False);pd.DataFrame({ID:te[ID],TARGET:p.idxmax(1)}).to_csv("submission.csv",index=False)
Path("training_summary.json").write_text(json.dumps({"experiment":"autogluon_medium_bag5","oof_balanced_accuracy":score,"elapsed_minutes":(time.time()-t0)/60,"leaderboard":predictor.leaderboard(silent=True).to_dict("records")},indent=2,default=str));print(score)
